# Metadata-first virality demo

Цель: показать, как **не скачивать десятки/сотни Shorts**, а сначала ранжировать их по публичным метаданным.

Пайплайн: `API metadata → score → repeated snapshots → shortlist → 1–3 downloads → Hypit`.

> По умолчанию тут синтетические demo-данные. Они нужны только чтобы воспроизводимо показать методику и графики.


In [ ]:
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
N = 80
now = datetime.now(timezone.utc)

age_hours = rng.uniform(3, 168, N)
subscribers = np.maximum(100, rng.lognormal(9.2, 1.6, N)).astype(int)
momentum = rng.lognormal(7.2, 1.2, N)
views = np.maximum(100, (momentum * age_hours * rng.uniform(.6, 1.4, N))).astype(int)
like_rate_raw = np.clip(rng.normal(.045, .022, N), .003, .16)
comment_rate_raw = np.clip(rng.lognormal(-5.8, .8, N), .0001, .025)

df = pd.DataFrame({
    "video_id": [f"demo_{i:03d}" for i in range(N)],
    "published_at": [now - timedelta(hours=float(h)) for h in age_hours],
    "duration_seconds": rng.integers(18, 121, N),
    "views": views,
    "likes": (views * like_rate_raw).astype(int),
    "comments": (views * comment_rate_raw).astype(int),
    "channel_subscribers": subscribers,
})
df.head()


## Метрики без скачивания

На первом проходе считаем:
- `age_hours`
- `views_per_hour_lifetime`
- `like_rate`
- `comment_rate`
- `engagement_rate`
- `views_per_subscriber` — breakout относительно размера канала

Score ниже — не «вероятность стать вирусным», а дешёвый ranking proxy внутри текущей выборки.


In [ ]:
now_ts = pd.Timestamp.now(tz="UTC")
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
df["age_hours"] = ((now_ts - df["published_at"]).dt.total_seconds()/3600).clip(lower=1)
df["views_per_hour_lifetime"] = df["views"] / df["age_hours"]
df["like_rate"] = df["likes"] / df["views"].clip(lower=1)
df["comment_rate"] = df["comments"] / df["views"].clip(lower=1)
df["engagement_rate"] = (df["likes"] + df["comments"]) / df["views"].clip(lower=1)
df["views_per_subscriber"] = df["views"] / df["channel_subscribers"].clip(lower=1)

def pct(s, higher=True):
    r = s.rank(pct=True)
    return r if higher else 1-r

df["metadata_virality_score"] = (
    .45*pct(np.log1p(df["views_per_hour_lifetime"])) +
    .20*pct(df["like_rate"]) +
    .10*pct(df["comment_rate"]) +
    .15*pct(np.log1p(df["views_per_subscriber"])) +
    .10*pct(df["age_hours"], higher=False)
)

ranked = df.sort_values("metadata_virality_score", ascending=False).reset_index(drop=True)
ranked[["video_id","views","age_hours","views_per_hour_lifetime","like_rate","comment_rate","views_per_subscriber","metadata_virality_score"]].head(12)


In [ ]:
top = ranked.head(15).sort_values("metadata_virality_score")
plt.figure(figsize=(10,6))
plt.barh(top["video_id"], top["metadata_virality_score"])
plt.xlabel("Metadata virality score")
plt.title("Top candidates by metadata-only score")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(ranked["views_per_hour_lifetime"], ranked["engagement_rate"]*100, alpha=.7)
plt.xscale("log")
plt.xlabel("Lifetime views/hour (log)")
plt.ylabel("Engagement rate, %")
plt.title("Velocity vs engagement")
plt.tight_layout()
plt.show()


## Самое ценное — повторные metadata snapshots

`views / age_hours` — средняя скорость за всю жизнь ролика. Лучше просто через 2–6 часов снова запросить `views/likes/comments`.

Тогда без скачивания получаем:

`delta_views_per_hour = (views_t2 - views_t1) / hours`

и можем видеть реальный текущий momentum.


In [ ]:
snap = ranked.head(12).copy()
snap["views_t0"] = snap["views"]
snap["views_t1"] = (snap["views_t0"] + snap["views_per_hour_lifetime"]*rng.uniform(.4,2.2,len(snap))*3).astype(int)
snap["views_t2"] = (snap["views_t1"] + snap["views_per_hour_lifetime"]*rng.uniform(.3,3.0,len(snap))*3).astype(int)
snap["growth_0_1"] = (snap["views_t1"]-snap["views_t0"])/3
snap["growth_1_2"] = (snap["views_t2"]-snap["views_t1"])/3
snap["acceleration"] = snap["growth_1_2"]-snap["growth_0_1"]

trend = snap.sort_values("growth_1_2", ascending=False).head(6)
plt.figure(figsize=(10,6))
for _, row in trend.iterrows():
    plt.plot([0,3,6], [row.views_t0,row.views_t1,row.views_t2], marker="o", label=row.video_id)
plt.xlabel("Hours since first snapshot")
plt.ylabel("Views")
plt.title("Current momentum from repeated metadata")
plt.legend()
plt.tight_layout()
plt.show()


## Download funnel

Для 80 кандидатов разумный MVP:

- 80: только API metadata
- 10: shortlist
- 5: cheap enrichment (title/description/tags/thumbnail/captions, если доступны)
- 1–3: полное видео + Hypit

Hypit **не нужен для metadata virality score**. Его место — после shortlist: разобрать reference video в workflow и делать контролируемые варианты hook / captions / B-roll / CTA / pacing.

Дальше уже на **своих** опубликованных вариантах можно забирать YouTube Analytics (retention, average view %, shares) и связывать результат с параметрами Hypit workflow.


In [ ]:
funnel = pd.DataFrame({
    "stage": ["API metadata","Shortlist","Cheap enrichment","Full video / Hypit"],
    "count": [len(ranked),10,5,3],
})
plt.figure(figsize=(9,5))
plt.bar(funnel["stage"], funnel["count"])
plt.ylabel("Candidates")
plt.title("Full downloads only for finalists")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

print(f"Full downloads avoided: {1-3/len(ranked):.1%}")
